# Superconducting Qubits

## A closer look at the modality that powers IBM, Google, and most cloud-accessible devices today. The plan: from a quantum LC oscillator, to a Josephson-junction-based anharmonic oscillator (the transmon), to how single- and two-qubit gates actually happen on hardware, to how measurement is performed.

# 1. The LC oscillator and why it isn't a qubit

## A capacitor $C$ in parallel with an inductor $L$ has Hamiltonian

$$ \Large  H_{\text{LC}} = \frac{Q^2}{2C} + \frac{\Phi^2}{2L}, $$

## where $Q$ is the charge on the capacitor and $\Phi$ the flux through the inductor. This is a harmonic oscillator with frequency $\omega_0 = 1/\sqrt{LC}$. Quantized, it has equally-spaced energy levels $E_n = \hbar\omega_0(n + 1/2)$.

## Equal spacing is the *problem*: a microwave pulse resonant with $\omega_0$ drives transitions between *all* adjacent levels — it cannot selectively couple $|0\rangle \leftrightarrow |1\rangle$ without leaking population into $|2\rangle, |3\rangle, \dots$. A qubit requires **anharmonicity**: the $|0\rangle \to |1\rangle$ transition must differ in frequency from $|1\rangle \to |2\rangle$.

# 2. The Josephson junction = nonlinear inductor

## A **Josephson junction** is a thin (~ 1 nm) insulating barrier between two superconductors. Cooper pairs tunnel coherently across it, with current-phase relation

$$ \Large  I = I_c \sin\varphi, \qquad V = \frac{\hbar}{2e}\dot\varphi. $$

## Combining these gives a *nonlinear* inductance $L_J(\varphi) = \Phi_0 / (2\pi I_c \cos\varphi)$, where $\Phi_0 = h/2e$ is the flux quantum.

## Replace the linear $L$ in the LC oscillator with a Josephson junction and the Hamiltonian becomes

$$ \Large  H = 4 E_C\, n^2 - E_J \cos\varphi, $$

## with charging energy $E_C = e^2/(2C)$ and Josephson energy $E_J = I_c \Phi_0 / 2\pi$, $n$ = number of Cooper pairs on the island. The cosine potential is anharmonic — and that is what we wanted.

# 3. The transmon regime

## **Transmon** (Koch et al., 2007) = the same circuit operated in the regime $E_J / E_C \gg 1$ (typically 50–100). A large shunt capacitor flattens the dependence on charge offsets, *suppressing charge noise exponentially* — at the cost of slightly weaker anharmonicity.

## In this regime, the two lowest levels of the cosine potential are well-approximated by a weakly-anharmonic oscillator:

$$ \Large  E_n \;\approx\; \hbar\omega_q\,n \;-\; \frac{E_C}{2} n(n-1), \qquad \omega_q \approx \sqrt{8 E_J E_C}/\hbar. $$

## - $\omega_q / 2\pi \sim 4{-}6$ GHz — set by $E_J, E_C$ at design time.
## - Anharmonicity $\alpha = E_{12} - E_{01} \approx -E_C \sim -200$ MHz. Negative, so $|1\rangle \to |2\rangle$ is at *lower* frequency than $|0\rangle \to |1\rangle$.
## - Pulses with bandwidth $\ll |\alpha|$ stay safely in the qubit subspace. This bandwidth constraint is why single-qubit gates take ~20 ns and not 1 ns.

In [1]:
# Diagonalise the transmon Hamiltonian and look at the first few levels.
import numpy as np
import matplotlib.pyplot as plt

EC = 0.25            # GHz
EJ = 25 * EC         # transmon regime: EJ/EC = 25

N = 60               # truncate charge basis at +/- N
n = np.diag(np.arange(-N, N + 1))
cos_phi = 0.5 * (np.diag(np.ones(2*N), 1) + np.diag(np.ones(2*N), -1))

H = 4 * EC * n @ n - EJ * cos_phi
eigs = np.linalg.eigvalsh(H)[:6]

print('Lowest 6 levels (GHz):', np.round(eigs, 4))
print(f'omega_q (0->1) = {eigs[1] - eigs[0]:.4f} GHz')
print(f'omega   (1->2) = {eigs[2] - eigs[1]:.4f} GHz')
print(f'anharmonicity  = {(eigs[2] - eigs[1]) - (eigs[1] - eigs[0]):.4f} GHz  (~ -EC)')

Lowest 6 levels (GHz): [-4.5472 -1.2832  1.6544  4.3418  5.854   9.5406]
omega_q (0->1) = 3.2640 GHz
omega   (1->2) = 2.9377 GHz
anharmonicity  = -0.3263 GHz  (~ -EC)


# 4. Single-qubit gates: microwave pulses

## A capacitively-coupled microwave drive at the qubit frequency $\omega_q$ implements

$$ \Large  H_{\text{drive}} = \Omega(t) \cos(\omega_q t + \varphi)\, \hat n. $$

## In the rotating frame, this reduces to

$$ \Large  H_{\text{rot}} = \tfrac{\Omega(t)}{2}\big(\cos\varphi\, X + \sin\varphi\, Y\big), $$

## a controllable rotation around an axis in the $XY$-plane. So:

## - $X(\pi/2)$ — apply a Gaussian "DRAG" pulse of area $\pi/2$ at phase $0$.
## - $Y(\pi/2)$ — same pulse at phase $\pi/2$.
## - $Z(\theta)$ — *not* a physical pulse. Implemented as a **virtual-Z**: shift the phase of all subsequent pulses by $-\theta$ (McKay et al., 2017). Free, instantaneous, error-free.

## The **DRAG** (Derivative Removal by Adiabatic Gate) pulse shape compensates leakage to $|2\rangle$ by adding a $\sin$-shaped quadrature component proportional to the derivative of the envelope.

# 5. Two-qubit gates

## Three popular families on hardware:

## ### Cross-resonance (CR) — IBM heavy-hex devices

## Two transmons at *different* frequencies are coupled via a bus resonator. Drive qubit $A$ at the frequency of qubit $B$; the off-resonant drive on $A$ creates a small effective $ZX$ interaction on $B$:

$$ \Large  H_{\text{CR}} \;\sim\; \omega_{CR}\, Z_A X_B + (\text{spurious terms}). $$

## With echo pulses to cancel the spurious terms, this synthesises a CNOT in ~200–500 ns.

## ### Tunable-coupler iSWAP / CZ — Google Sycamore, Willow

## A tunable inductive coupler between two transmons is briefly turned on, exchanging excitations between $|01\rangle \leftrightarrow |10\rangle$ (an **iSWAP**) or producing a **CZ** by going via the $|11\rangle \leftrightarrow |20\rangle$ avoided crossing. Gate time ~25 ns; the speed comes from leaving the coupler off most of the time, so qubits are well-isolated when idle.

## ### Parametric / Sycamore-style

## Modulate the coupler at the *difference* frequency to drive the same exchange — gives more flexibility in scheduling.

## All three give entangling gates with $\sim 99.5{-}99.9\%$ fidelity today. Two-qubit error is the dominant cost in fault-tolerant resource estimates.

# 6. Readout: dispersive measurement

## Each qubit is coupled to a dedicated **readout resonator** (a microwave cavity). In the *dispersive* regime — qubit detuning $\Delta = \omega_q - \omega_r$ much larger than the coupling $g$ — the resonator frequency shifts by $\pm \chi$ depending on the qubit state:

$$ \Large  \omega_r \;\to\; \omega_r + \chi \langle Z \rangle. $$

## Send a low-power probe tone through the resonator and measure the transmitted/reflected amplitude and phase. The result clusters into two distinguishable blobs in the IQ plane — $|0\rangle$ vs $|1\rangle$ — and is digitised into a measurement bit.

## **Numbers:** readout time ~ 200 ns to 1 μs; single-shot fidelity ~ 95–99%. Readout is often the **slowest and noisiest** part of a circuit.

## Crucially, dispersive readout is **QND** (Quantum Non-Demolition) in principle: it projects into the computational basis without further perturbing the post-measurement state. This is what enables **mid-circuit measurement** and feedback used in quantum error correction.

# 7. The full control stack

## A working superconducting machine is a layered system:

## - **Chip** (10 mK in a dilution refrigerator): transmons + couplers + readout resonators, lithographically patterned, wire-bonded to a sample holder.
## - **Cryogenic wiring**: attenuators, filters, and circulators at each cooling stage to deliver clean microwave pulses without thermal noise.
## - **Room-temperature electronics**: arbitrary waveform generators (AWGs), local oscillators, mixers, ADCs.
## - **FPGA / RFSoC**: real-time pulse sequencing, mid-circuit feedback, ADC readout demodulation.
## - **Classical compute**: schedule pulses, run calibrations, post-process measurement results.
## - **User-facing software**: Qiskit Pulse, Q-CTRL, Quantum Machines QUA, etc.

## When a Qiskit circuit is "submitted", what really happens: the circuit is transpiled to the device's native gates, scheduled into a timeline of pulses, sent to the AWGs, played at 10 mK, sampled by readout, demodulated by the FPGA, returned as counts — typically in tens of milliseconds per shot.

# Recap

## - **Transmon** = Josephson-junction LC oscillator in the $E_J \gg E_C$ regime: weakly anharmonic, charge-noise insensitive.
## - Frequencies ~ 5 GHz, anharmonicity ~ −200 MHz.
## - **Single-qubit gates** = microwave DRAG pulses; **$R_z$ is virtual** (just a phase shift on subsequent pulses).
## - **Two-qubit gates** = cross-resonance, tunable-coupler iSWAP/CZ, or parametric drives.
## - **Readout** = dispersive — qubit pulls the resonator frequency; transmitted tone tells the story.
## - The control stack runs from a dilution fridge to room-temperature FPGAs to a Python SDK.

## Next: how the device's *topology* (which qubits are physically connected) and *native gate set* (which operations are pulse-implemented) shape what circuits can actually be run.